In [7]:
from sentence_transformers import SentenceTransformer

# Load the local model (it will download about 80MB the first time you run this)
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# The legal text we want to mathematically encode
text_to_embed = "La période d'essai pour un CDI est de 3 mois pour les cadres."

# Generate the vector locally on your ThinkPad's CPU
vector = embedding_model.encode(text_to_embed)

print(f"Dimension of the vector (Length): {len(vector)}")
print(f"First 5 numbers of the embedding:\n{vector[:5]}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimension of the vector (Length): 384
First 5 numbers of the embedding:
[-0.17237613 -0.01570499  0.3107673  -0.46348062  0.00488091]


In [8]:
from elasticsearch import Elasticsearch

# Connect to your local Docker container
es_client = Elasticsearch('http://localhost:9200')

# Name of the new vector database index
index_name = "juris_maroc_vectors"

# Define the structure of the database
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "keyword"},
            # This is the AI upgrade: a dedicated field for the 384 math coordinates
            "text_vector": {
                "type": "dense_vector",
                "dims": 384,
                "index": True,
                "similarity": "cosine"
            }
        }
    }
}

# Delete the index if it already exists (useful for iterative testing)
if es_client.indices.exists(index=index_name):
    es_client.indices.delete(index=index_name)

# Create the new vector index
es_client.indices.create(index=index_name, body=index_settings)

print(f"Index '{index_name}' created successfully with Vector Search capabilities.")

/tmp/ipykernel_9357/2159062940.py:35: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body=index_settings)


Index 'juris_maroc_vectors' created successfully with Vector Search capabilities.


In [9]:
from elasticsearch import Elasticsearch

# Connect to your local Docker container
es_client = Elasticsearch('http://localhost:9200')

# Name of the new vector database index
index_name = "juris_maroc_vectors"

# Define the structure of the database
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "keyword"},
            # This is the AI upgrade: a dedicated field for the 384 math coordinates
            "text_vector": {
                "type": "dense_vector",
                "dims": 384,
                "index": True,
                "similarity": "cosine"
            }
        }
    }
}

# Delete the index if it already exists (useful for iterative testing)
if es_client.indices.exists(index=index_name):
    es_client.indices.delete(index=index_name)

# Create the new vector index
es_client.indices.create(index=index_name, body=index_settings)

print(f"Index '{index_name}' created successfully with Vector Search capabilities.")

/tmp/ipykernel_9357/2159062940.py:35: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body=index_settings)


Index 'juris_maroc_vectors' created successfully with Vector Search capabilities.


In [10]:
import json
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
from tqdm.auto import tqdm  # This gives us a nice progress bar

# 1. Connect to Elasticsearch & Load Model
es_client = Elasticsearch('http://localhost:9200')
index_name = "juris_maroc_vectors"
embedding_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1")

# 2. Load the Moroccan Labor Code data
with open('labor_code_chunks.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

# 3. Loop, Embed, and Index
for doc in tqdm(documents, desc="Embedding and Indexing Laws"):
    # Calculate the vector using the correct 'content' key
    vector = embedding_model.encode(doc['content'])

    # Prepare the payload mapping your exact JSON keys
    es_doc = {
        "text": doc['content'],
        "article_number": doc.get('article_number', 'N/A'),
        "id": doc.get('id', ''),
        "source": doc.get('source', ''),
        "text_vector": vector.tolist()
    }

    # Inject into the database
    es_client.index(index=index_name, document=es_doc)

print("✅ All legal documents are successfully embedded and indexed!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding and Indexing Laws:   0%|          | 0/587 [00:00<?, ?it/s]

✅ All legal documents are successfully embedded and indexed!


In [6]:
# Look at the keys of the very first chunk in your JSON
print(documents[0].keys())

dict_keys(['id', 'article_number', 'content', 'source', 'language'])


In [11]:
# 1. The user asks a natural question
user_query = "Comment je peux démissionner de mon travail ?"

# 2. Translate the question into a 384-dimensional vector arrow
query_vector = embedding_model.encode(user_query).tolist()

# 3. Build the Elasticsearch KNN payload
search_query = {
    "knn": {
        "field": "text_vector",
        "query_vector": query_vector,
        "k": 3,               # Return the top 3 closest arrows (matches)
        "num_candidates": 100 # Optimization: Only calculate exact angles for the 100 closest candidates
    },
    "_source": ["article_number", "text"] # Only return these fields to save bandwidth
}

# 4. Execute the search against your local container
response = es_client.search(
    index=index_name,
    knn=search_query["knn"],
    _source=search_query["_source"]
)

# 5. Display the matching articles and their Cosine Similarity scores
print(f"Question: '{user_query}'\n" + "-"*50)
for hit in response['hits']['hits']:
    score = hit['_score']
    article = hit['_source'].get('article_number', 'N/A')
    text = hit['_source']['text']

    print(f"Score de similarité: {score}")
    print(f"Article: {article}")
    print(f"Extrait: {text[:200]}...\n")

Question: 'Comment je peux démissionner de mon travail ?'
--------------------------------------------------
Score de similarité: 0.77681625
Article: 218
Extrait: Article 218 
Il peut être décidé que le jour férié soit rémunéré comme temps de 
travail effectif....

Score de similarité: 0.7731632
Article: 231
Extrait: Article 231 
Sauf dispositions plus favorables du contrat de travail, de la 
convention collective de travail, du règlement intérieur ou des usages, 
tout salarié a droit, après six mois de service co...

Score de similarité: 0.76560414
Article: 352
Extrait: Article 352 
Sont considérées comme périodes de travail effectif et ne peuvent 
être déduites de la durée des services entrant en ligne de compte pour 
l'attribution de la prime d'ancienneté : 
1. les...



In [ ]:
from sentence_transformers import SentenceTransformer
from elasticsearch import Elasticsearch
import json
from tqdm.auto import tqdm

# 1. Force the notebook to use the Multilingual model
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
es_client = Elasticsearch('http://localhost:9200')
index_name = "juris_maroc_vectors"

# 2. DESTROY the old database to clear out the English vectors
if es_client.indices.exists(index=index_name):
    es_client.indices.delete(index=index_name)

# Recreate the clean index
es_client.indices.create(index=index_name, body={
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "article_number": {"type": "keyword"},
            "text_vector": {"type": "dense_vector", "dims": 384, "index": True, "similarity": "cosine"}
        }
    }
})

# 3. Re-ingest the data using the new French-aware model
with open('labor_code_chunks.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

for doc in tqdm(documents, desc="Embedding Multilingual Laws"):
    es_doc = {
        "text": doc['content'],
        "article_number": doc.get('article_number', 'N/A'),
        "text_vector": embedding_model.encode(doc['content']).tolist()
    }
    es_client.index(index=index_name, document=es_doc)

# 4. Search again (Ensuring the query is embedded with the new model)
user_query = "Quelles sont les conditions de la démission et la rupture de contrat ?"
query_vector = embedding_model.encode(user_query).tolist()

response = es_client.search(
    index=index_name,
    knn={"field": "text_vector", "query_vector": query_vector, "k": 3, "num_candidates": 100},
    _source=["article_number", "text"]
)

print(f"\nQuestion: '{user_query}'\n" + "-"*50)
for hit in response['hits']['hits']:
    print(f"Score: {hit['_score']}")
    print(f"Article: {hit['_source'].get('article_number', 'N/A')}")
    print(f"Extrait: {hit['_source']['text'][:200]}...\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_6228/4087815361.py:16: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  es_client.indices.create(index=index_name, body={


Embedding Multilingual Laws:   0%|          | 0/587 [00:00<?, ?it/s]